# Beam-search eval summary (`eval_results/beam`)

For every CSV in `eval_results/beam` we summarize three reward quantities, **averaged over all proteins/sequences in the file**:

- **ddg align** — the alignment (training) oracle reward. The `spec_trajectory` column records this reward *before* and *after* the spectral feedback step, so it gives us a raw delta and a % improvement. `spec_trajectory[0]` is the initial sampled reward, `spec_trajectory[-1]` is the post-feedback reward (equal to the `ddg` column).
- **ddg eval** — the held-out evaluation oracle reward on the final sequence (`ddg_eval` column). Only the final value is stored, so we report its mean per experiment.
- **spectral trajectory** — the reward trajectory itself (`spec_trajectory`), from which the raw delta (`end - start`) and percentage improvement (`(end - start) / |start| * 100`) are computed.

The "% improvement" is computed per sequence and then averaged across all sequences in the file.

In [1]:
import ast
import glob
import os
import re

import numpy as np
import pandas as pd

BEAM_DIR = "eval_results/beam"


def parse_experiment_label(fname):
    """Pull the distinguishing hyper-parameters out of a beam CSV filename."""
    fields = {}
    for key in ["N", "W", "feedbacksteps", "feedbackmethod", "maxspecorder",
                "masks", "stepsperlevel", "tilt_beta"]:
        m = re.search(rf"{key}=([^_]+)", fname)
        if m:
            fields[key] = m.group(1)
    parts = []
    if "N" in fields:
        parts.append(f"N={fields['N']}")
    if "W" in fields:
        parts.append(f"W={fields['W']}")
    if "stepsperlevel" in fields:
        parts.append(f"spl={fields['stepsperlevel']}")
    return ", ".join(parts) if parts else fname


def summarize_file(path):
    df = pd.read_csv(path)
    traj = df["spec_trajectory"].apply(ast.literal_eval)

    start = traj.apply(lambda t: float(t[0]))   # ddg-align reward before feedback
    end = traj.apply(lambda t: float(t[-1]))    # ddg-align reward after feedback
    raw_delta = end - start
    pct_improvement = raw_delta / start.abs() * 100.0

    return {
        "experiment": parse_experiment_label(os.path.basename(path)),
        "file": os.path.basename(path),
        "n_proteins": int(df["protein_name"].nunique()),
        "n_seqs": int(len(df)),
        # ddg align (alignment/training oracle reward)
        "ddg_align_start_mean": start.mean(),
        "ddg_align_end_mean": end.mean(),
        "ddg_align_raw_delta_mean": raw_delta.mean(),
        "ddg_align_pct_improve_mean": pct_improvement.mean(),
        "ddg_align_final_mean": df["ddg"].mean(),
        # ddg eval (held-out evaluation oracle reward, final sequence only)
        "ddg_eval_mean": df["ddg_eval"].mean(),
        "ddg_eval_std": df["ddg_eval"].std(),
    }


files = sorted(glob.glob(os.path.join(BEAM_DIR, "*.csv")))
summary = pd.DataFrame([summarize_file(f) for f in files])
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
pd.set_option("display.max_colwidth", 120)
summary

,experiment,file,n_proteins,n_seqs,ddg_align_start_mean,ddg_align_end_mean,ddg_align_raw_delta_mean,ddg_align_pct_improve_mean,ddg_align_final_mean,ddg_eval_mean,ddg_eval_std
0,"N=10, W=1, spl=1",pretrained_test_ddg_beam_N=10_feedbacksteps=1_feedbackmethod=spectral_maxspecorder=20_masks=8192_rmax=True__W=1_step...,12,24,0.5731,0.6713,0.0982,34.7855,0.6701,0.6560,1.2246
1,"N=10, W=2, spl=1",pretrained_test_ddg_beam_N=10_feedbacksteps=1_feedbackmethod=spectral_maxspecorder=20_masks=8192_rmax=True__W=2_step...,12,24,0.6113,0.6929,0.0817,15.3616,0.6926,0.7914,0.9655
2,"N=5, W=1, spl=1",pretrained_test_ddg_beam_N=5_feedbacksteps=1_feedbackmethod=spectral_maxspecorder=20_masks=8192_rmax=True__W=1_steps...,12,24,0.5450,0.5970,0.0520,18.0288,0.5979,0.7073,1.2046


In [2]:
for _, row in summary.iterrows():
    print(f"=== {row['experiment']}  ({row['n_proteins']} proteins, {row['n_seqs']} sequences) ===")
    print(f"  ddg align initial : {row['ddg_align_start_mean']:.4f}")
    print(f"  ddg align  : {row['ddg_align_start_mean']:.4f} -> {row['ddg_align_end_mean']:.4f}"
          f"  (raw delta {row['ddg_align_raw_delta_mean']:+.4f}, "
          f"{row['ddg_align_pct_improve_mean']:+.2f}% improvement)")
    print(f"  ddg eval   : {row['ddg_eval_mean']:.4f}  (+/- {row['ddg_eval_std']:.4f}, final sequence)")
    print()

report = summary[[
    "experiment", "n_proteins", "n_seqs",
    "ddg_align_start_mean", "ddg_align_raw_delta_mean", "ddg_align_pct_improve_mean",
    "ddg_align_final_mean", "ddg_eval_mean",
]].rename(columns={
    "ddg_align_start_mean": "align_initial",
    "ddg_align_raw_delta_mean": "align_raw_delta",
    "ddg_align_pct_improve_mean": "align_%_improve",
    "ddg_align_final_mean": "align_final",
    "ddg_eval_mean": "eval_final",
})
report

=== N=10, W=1, spl=1  (12 proteins, 24 sequences) ===
  ddg align initial : 0.5731
  ddg align  : 0.5731 -> 0.6713  (raw delta +0.0982, +34.79% improvement)
  ddg eval   : 0.6560  (+/- 1.2246, final sequence)

=== N=10, W=2, spl=1  (12 proteins, 24 sequences) ===
  ddg align initial : 0.6113
  ddg align  : 0.6113 -> 0.6929  (raw delta +0.0817, +15.36% improvement)
  ddg eval   : 0.7914  (+/- 0.9655, final sequence)

=== N=5, W=1, spl=1  (12 proteins, 24 sequences) ===
  ddg align initial : 0.5450
  ddg align  : 0.5450 -> 0.5970  (raw delta +0.0520, +18.03% improvement)
  ddg eval   : 0.7073  (+/- 1.2046, final sequence)



,experiment,n_proteins,n_seqs,align_initial,align_raw_delta,align_%_improve,align_final,eval_final
0,"N=10, W=1, spl=1",12,24,0.5731,0.0982,34.7855,0.6701,0.6560
1,"N=10, W=2, spl=1",12,24,0.6113,0.0817,15.3616,0.6926,0.7914
2,"N=5, W=1, spl=1",12,24,0.5450,0.0520,18.0288,0.5979,0.7073
